# Build a hierarchical spatial query taxonomy from embeddings

In [63]:
import pandas as pd
import numpy as np
import umap
import csv
from pathlib import Path
import hdbscan
import json

In [3]:
# Load raw queries
queries = pd.read_csv("interim/queries.csv.zip")

# Load query embeddings
xb = np.load("interim/query_emb.npy").astype('float32')  # (n_queries, 384)

# Load classifier results
classifier_results = pd.read_csv('output/queries-classified.csv.zip')

In [4]:
queries.shape

(1010916, 2)

In [5]:
spatial_idx = classifier_results.is_spatial_pred.eq(1)

In [127]:
queries[queries.query_text == "earthquake last week in india"]

,query_id,query_text
176910,177371,earthquake last week in india


In [129]:
queries[spatial_idx][queries.query_text.str.contains('which county is') ].sample(10)

/var/folders/w6/kmxyhb092_3gsgy99f9_t01r0000gn/T/ipykernel_25880/986186406.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  queries[spatial_idx][queries.query_text.str.contains('which county is') ].sample(10)


,query_id,query_text
791631,1007877,which county is harrison ny located
998632,1007837,which county is cincinnati oh located
327931,394665,in which county is north hollywood california
693026,1007993,which county is wellesley ma in
119654,1104626,"which county is healdsburg,ca"
163042,394662,in which county is merrimack street new hampshire
644825,1007969,which county is sheridan ar in
202418,1007813,"which county is black station, ca"
281662,1140090,which county is hillingdon in
689429,1140084,which county is north va in


## UMAP + HDBSCAN

### Run grid search to find best parameters

1. We reduce embedding dimention with Umap before clustering (384 is too many for hsbscan)
2. No right way of doing it, but let's try 5-10-15 dimensions
3. Try a bunch of other params as well (grid search-like)

In [6]:
xb_spatial = xb[spatial_idx]
xb_spatial.shape

(104288, 384)

In [6]:
def log(path='output/umap_hdbscan_runs.csv', config_id=None):
    
    log_path = Path(path)
    write_header = not log_path.exists()
    
    with log_path.open('a', newline='') as f:
        writer = csv.writer(f)
    
        if write_header:
            writer.writerow([
                'config_id',
                'seed',
                'umap_dim',
                'umap_neighbours',
                'umap_min_dist',
                'min_cluster_size',
                'min_samples',
                'dbcv',
                'noise_frac',
                'n_clusters',
                'median_cluster_size',
                'cluster_selection_method'
            ])
    
        writer.writerow([
            config_id,
            seed,
            umap_dim,
            umap_neighbours,
            umap_min_dist,
            min_cluster_size,
            min_samples,
            dbcv,
            noise,
            n_clusters,
            median_size,
            cluster_selection_method
        ])

### Run grid search to find top-5 parameters

In [11]:
%%time

cluster_selection_method = 'eom' # by default, let's search the grid with eom

for umap_dim in (5, 10, 15):
    for umap_neighbours in (13, 30, 50):
        for umap_min_dist in (0.0, 0.1):

            for min_cluster_size in (25, 50, 100, 200):
                for min_samples in (None, 5, 10):

                    print('----')
                    print('UMAP n dims:', umap_dim)
                    print('UMAP n neighbours:', umap_neighbours)
                    print('UMAP min dist:', umap_min_dist)
                    print('HDBSCAN min cluster size:', min_cluster_size)
                    print('HDBSCAN min samples:', min_samples)
            
                    reducer = umap.UMAP(
                        n_components=umap_dim,
                        n_neighbors=umap_neighbours,
                        min_dist=umap_min_dist,
                        metric='cosine',
                        n_jobs=1, # -1 for all cores for faster processing but then cannot use random seed...
                        random_state=42
                    )
                    
                    xb_spatial_low = reducer.fit_transform(xb_spatial)
            
                    clusterer = hdbscan.HDBSCAN(
                        min_cluster_size=min_cluster_size,
                        min_samples=min_samples,
                        metric='euclidean',
                        cluster_selection_method=cluster_selection_method,
                        gen_min_span_tree=True
                    )

                    # Fit the model
                    labels = clusterer.fit_predict(xb_spatial_low)

                    # Calculate quality metrics
                    dbcv = clusterer.relative_validity_
                    noise = (labels == -1).mean() # -1 aka invalid/noise
                    
                    n_clusters = len(set(labels)) - (1 if -1 in labels else 0) # exclude noise -1 cluster
                    
                    vc = pd.Series(labels[labels != -1]).value_counts()
                    median_size = vc.median() if len(vc) else 0

                    print(f'DBCV Score: {dbcv}')
                    print(f'% invalid/noise/-1 clusters: {noise}')
                    print(f'# clusters: {n_clusters}')
                    print(f'median cluster size: {median_size}')
                    print('*****************')

                    # Log to the file
                    log()

----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 25
HDBSCAN min samples: None
DBCV Score: 0.2416853653441588
% invalid/noise/-1 clusters: 0.45013807916538817
# clusters: 480
median cluster size: 60.5
*****************
----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 25
HDBSCAN min samples: 5
DBCV Score: 0.19119169224845756
% invalid/noise/-1 clusters: 0.39415848419760663
# clusters: 699
median cluster size: 49.0
*****************
----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 25
HDBSCAN min samples: 10
DBCV Score: 0.19758450290896498
% invalid/noise/-1 clusters: 0.42727830622890456
# clusters: 622
median cluster size: 51.0
*****************
----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 50
HDBSCAN min samples: None
DBCV Score: 0.33264898840420576
% invalid/noise/-1 clusters: 0.42451672292114145
# clusters: 195
median cluster size: 1

## Check how stable clustering is

- Umap is stochastic
- Let's rerun umap+hdbscan for top config 5 times each to see how `dbcv` and other qa metrics differ
- We then have more confidence in picking the best params
- top-10 config gets dbcv >= 0.355, which is a nice initial threshold (and all noise < 0.42; between 43 and 174 clusters)

In [73]:
top_configs = (
    pd.read_csv('output/umap_hdbscan_runs.csv')
        .sort_values('dbcv', ascending=False)
        .head(10)
)

top_configs.round(3).drop(columns=['seed', 'cluster_selection_method'])

,umap_dim,umap_neighbours,umap_min_dist,min_cluster_size,min_samples,dbcv,noise_frac,n_clusters,median_cluster_size
202,15,50,0.0,200,5.0,0.422,0.334,71,576.0
57,5,50,0.0,200,NaN,0.418,0.360,48,554.0
201,15,50,0.0,200,NaN,0.390,0.351,48,570.0
106,10,30,0.0,200,5.0,0.384,0.329,78,426.5
131,10,50,0.0,200,10.0,0.378,0.334,75,517.0
203,15,50,0.0,200,10.0,0.377,0.337,71,525.0
178,15,30,0.0,200,5.0,0.374,0.337,75,450.0
130,10,50,0.0,200,5.0,0.365,0.343,77,543.0
195,15,50,0.0,50,NaN,0.356,0.420,174,147.5
33,5,30,0.0,200,NaN,0.355,0.382,43,565.0


In [24]:
%%time

for i, config in top_configs.iterrows():

    for cluster_selection_method in ['eom']:# 'leaf': # leaf produces slightly more clusters, slightly smaller clusters, but higher noise and lower dbcv, so let's ignore.
        
        for seed in [42, 0, 1, 2, 3, 4]: # check 42 again to make sure we can reproduce

            # get params into vars so logger knows what to write
            umap_dim = config['umap_dim']
            umap_neighbours = config['umap_neighbours']
            umap_min_dist = config['umap_min_dist']
            min_cluster_size = config['min_cluster_size']
            min_samples = None if pd.isna(config['min_samples']) else int(config['min_samples']) # np.nan and None (expected by hdbscan's min_samples) are not equivalent here so we need this extra check
    
            reducer = umap.UMAP(
                n_components=umap_dim,
                n_neighbors=umap_neighbours,
                min_dist=umap_min_dist,
                metric='cosine',
                n_jobs=1,
                random_state=seed
            )
    
            xb_spatial_low = reducer.fit_transform(xb_spatial)
                        
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples, 
                metric='euclidean',
                cluster_selection_method=cluster_selection_method,
                gen_min_span_tree=True
            )

            # Fit the model
            labels = clusterer.fit_predict(xb_spatial_low)

            # Calculate quality metrics
            dbcv = clusterer.relative_validity_
            noise = (labels == -1).mean() # label=-1 for invalid/noise
            
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0) # exclude noise (label=-1) cluster
            
            vc = pd.Series(labels[labels != -1]).value_counts()
            median_size = vc.median() if len(vc) else 0

            log('output/top_runs_different_seeds.csv', i)

CPU times: user 1h 46min 51s, sys: 59 s, total: 1h 47min 50s
Wall time: 1h 49min 4s


# Get cluster labels from the best performing set of umap+hdbscan params

## Let's summarise run consistency per each top config & which config to use

- Probably highest dbcv mean assuming it also has low noise fraction

In [74]:
top_runs = pd.read_csv('output/top_runs_different_seeds.csv')

top_runs_stats = top_runs.groupby('config_id').aggregate({
    'dbcv': ('max', 'mean', 'std'),
    'noise_frac': ('max', 'mean', 'std'),
    'n_clusters': ('max', 'mean', 'std')
}).round(3).sort_values([('dbcv', 'mean')], ascending=False)

top_runs_stats

dbcv               noise_frac               n_clusters           \
             max   mean    std        max   mean    std        max     mean   
config_id                                                                     
106        0.396  0.341  0.052      0.359  0.333  0.019         83   76.500   
131        0.382  0.338  0.049      0.378  0.348  0.019         80   75.000   
178        0.388  0.332  0.064      0.347  0.331  0.014         81   75.167   
201        0.390  0.332  0.051      0.405  0.370  0.020         51   47.833   
203        0.377  0.330  0.045      0.379  0.349  0.017         83   75.000   
202        0.422  0.324  0.055      0.359  0.347  0.010         79   75.500   
130        0.365  0.312  0.041      0.392  0.347  0.031         86   78.333   
195        0.356  0.311  0.029      0.436  0.427  0.007        180  177.167   
57         0.418  0.303  0.082      0.435  0.388  0.033         54   50.000   
33         0.355  0.292  0.044      0.439  0.403  0.023         48   45.000   

                  
             std  
config_id         
106        4.593  
131        3.033  
178        3.312  
201        1.941  
203        5.404  
202        3.082  
130        5.203  
195        3.488  
57         3.162  
33         1.789

In [75]:
top_runs_stats.to_excel('top10consistency.xlsx')

In [9]:
# Ok let's re-label top run again to get the labels
top_runs.query(f'config_id == {top_runs_stats.iloc[0].name}')

,config_id,seed,umap_dim,umap_neighbours,umap_min_dist,min_cluster_size,min_samples,dbcv,noise_frac,n_clusters,median_cluster_size,cluster_selection_method
18,106,42,10,30,0.0,200,5.0,0.383788,0.328906,78,426.5,eom
19,106,0,10,30,0.0,200,5.0,0.295722,0.305193,71,468.0,eom
20,106,1,10,30,0.0,200,5.0,0.380119,0.327027,73,455.0,eom
21,106,2,10,30,0.0,200,5.0,0.314430,0.359150,83,466.0,eom
22,106,3,10,30,0.0,200,5.0,0.275548,0.348036,80,436.0,eom
23,106,4,10,30,0.0,200,5.0,0.395582,0.331102,74,489.0,eom


In [10]:
# Let's pick the seed with the highest dbcv
top_run = top_runs.query(f'config_id == {top_runs_stats.iloc[0].name}').sort_values('dbcv', ascending=False).iloc[0]
top_run

config_id                        106
seed                               4
umap_dim                          10
umap_neighbours                   30
umap_min_dist                    0.0
min_cluster_size                 200
min_samples                      5.0
dbcv                        0.395582
noise_frac                  0.331102
n_clusters                        74
median_cluster_size            489.0
cluster_selection_method         eom
Name: 23, dtype: object

In [11]:
%%time

reducer = umap.UMAP(
    n_components=int(top_run.umap_dim),
    n_neighbors=int(top_run.umap_neighbours),
    min_dist=top_run.umap_min_dist,
    metric='cosine',
    n_jobs=1,
    random_state=int(top_run.seed)
)
    
xb_spatial_low = reducer.fit_transform(xb_spatial)
            
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=int(top_run.min_cluster_size),
    min_samples=int(top_run.min_samples), 
    metric='euclidean',
    cluster_selection_method='eom',
    gen_min_span_tree=True
)

# Get the lables
labels = clusterer.fit_predict(xb_spatial_low)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


CPU times: user 1min 24s, sys: 811 ms, total: 1min 25s
Wall time: 1min 26s


In [12]:
print( len(labels) )
assert len(set(labels)) == (top_run.n_clusters + 1) # adding -1 for noise cluster

104288


## Manually review clusters

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cdist

In [15]:
def analyze_clusters(queries, labels, embeddings, top_n_words=20, sample_size=10):
    df = pd.DataFrame({'query': queries, 'label': labels})
    unique_labels = sorted(df['label'].unique())

    # class-based tf–idf
    docs_per_class = (
        df.groupby('label')['query']
        .apply(lambda x: ' '.join(x))
        .reset_index()
        .sort_values('label')
        .reset_index(drop=True)
    )

    tfidf_model = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = tfidf_model.fit_transform(docs_per_class['query'])
    words = tfidf_model.get_feature_names_out()

    rows = []

    for i, label in enumerate(unique_labels):
        idx = df.index[df['label'] == label]
        cluster_embeddings = embeddings[idx]
        cluster_queries = df.loc[idx, 'query'].values

        # centroid representative
        centroid = cluster_embeddings.mean(axis=0).reshape(1, -1)
        distances = cdist(centroid, cluster_embeddings, 'cosine')[0]
        rep_query = cluster_queries[distances.argmin()]

        # top tf–idf terms
        row_data = tfidf_matrix.getrow(i).toarray()[0]
        top_idx = row_data.argsort()[-top_n_words:][::-1]
        top_terms = [words[j] for j in top_idx]

        # sample queries
        sample_queries = (
            df.loc[idx, 'query']
            .sample(min(len(idx), sample_size), random_state=42)
            .tolist()
        )

        rows.append({
            'label': label,
            'n_samples': len(idx),
            'representative_query': rep_query,
            'top_terms': top_terms,
            'sample_queries': sample_queries,
        })

    return pd.DataFrame(rows).sort_values('label').reset_index(drop=True)


def clusters_df_to_markdown(clusters_df):
    sections = []

    for _, row in clusters_df.iterrows():
        section = [
            f'## Cluster {row["label"]} ({row["n_samples"]} samples)',
            f'**Representative Query:** `{row["representative_query"]}`',
            f'**Top Terms:** {", ".join(row["top_terms"])}',
            '',
            '**Sample Queries:**',
            '\n'.join(f'- {q}' for q in row['sample_queries']),
            '',
            '---',
        ]
        sections.append('\n'.join(section))

    return '\n'.join(sections)

In [16]:
final_clusters = analyze_clusters(
    queries[spatial_idx].reset_index().query_text,
    labels,
    xb[spatial_idx]
)

In [18]:
final_clusters_markdown = clusters_df_to_markdown(final_clusters)

In [19]:
with open('output/cluster_review.md', 'w') as f:
    f.write(final_clusters_markdown)

In [22]:
final_clusters.filter([
    'label', 'n_samples', 'representative_query'
]).to_excel('output/cluster_review.xlsx', index=False)

## Group clusters where necessary

- Eg 'places in California', 'places in Texas' etc

#### Convert spreadsheet to vega JSON

In [76]:
data = (
    pd.read_excel('output/cluster_review_edits.xlsx')
    .dropna(subset=['group'])
)

In [77]:
data.drop_duplicates(subset=['title']).set_index('title').group

title
Countries and cities (worldwide)                 Referencing
Best time or season to visit                        Temporal
Contact details                                   Attributes
Costs and prices                                  Attributes
Which county (US)                                Referencing
Distances and driving times                        Relations
Celestial bodies                                 Referencing
Filming locations                                Referencing
Foods and cuisines                                Attributes
Biomes, plants, and animals                     Distribution
Languages spoken                                Distribution
Legal status or eligibility                       Attributes
Location and capactiy of casinos and resorts      Attributes
Location and characteristics of volcanoes         Attributes
Rivers                                           Referencing
Mountains, landforms (elevation)                  Attributes
Buildings, monumen

In [78]:
group_colours = {
    'Referencing': '#66c2a5',
    'Relations': '#fc8d62',
    'Attributes': '#8da0cb',
    'Distribution': '#e78ac3',
    'Temporal': '#a6d854',
    'Other': '#bbbbbb'
}

In [79]:
other_size = int(pd.read_excel('output/cluster_review_edits.xlsx').query('group.isna()').n_samples.sum())
other_size

33083

In [80]:
group_counts = data.groupby(['group']).n_samples.sum().reset_index()
group_counts

,group,n_samples
0,Attributes,17356.0
1,Distribution,5555.0
2,Referencing,27725.0
3,Relations,7047.0
4,Temporal,13522.0


In [86]:
leaf_counts = data.groupby(['group', 'title']).n_samples.sum().reset_index().assign(
    pc=lambda df_: (df_.n_samples / (df_.n_samples.sum() + other_size)*100).round(1)
)

leaf_counts

,group,title,n_samples,pc
0,Attributes,Bridges and tunnels,288.0,0.3
1,Attributes,"Buildings, monuments (height, location)",437.0,0.4
2,Attributes,Contact details,1248.0,1.2
3,Attributes,Costs and prices,8374.0,8.0
4,Attributes,Foods and cuisines,207.0,0.2
5,Attributes,Lakes and water bodies,498.0,0.5
6,Attributes,Legal status or eligibility,1006.0,1.0
7,Attributes,Location and capactiy of casinos and resorts,241.0,0.2
8,Attributes,Location and characteristics of volcanoes,403.0,0.4
9,Attributes,"Mountains, landforms (elevation)",1829.0,1.8


In [60]:
root = [{"id": "root", "name": "Spatial queries"}]

group_level = [{
    "id": row['group'],
    "name": row['group'],
    "parent": "root",
    "size": row['n_samples'],
    'colour': group_colours.get(row['group'], '#000000')
} for _, row in group_counts.sort_values('n_samples', ascending=False).iterrows() ]

leaf_level = [{
    "id": row['title'],
    "name": row['title'],
    "parent": row['group'],
    "size": row['n_samples'],
    'colour': group_colours.get(row['group'], '#000000')
} for _, row in leaf_counts.sort_values('n_samples', ascending=False).iterrows()]

other_bubble = [{
    'id': 'Other',
    'name': 'Misc',
    'parent': 'root',
    'size': other_size,
    'colour': group_colours.get('Other', '#000000')
}]

In [61]:
nodes = root + group_level + leaf_level + other_bubble

In [62]:
with open('visual/data.json', 'w') as f:
    f.write(json.dumps(nodes))